# Section 4 — Feature Engineering



## 4.1.1 Imports & configuration
Import libraries and set the primary column names and input filename.

In [2]:
import pandas as pd
import numpy as np
from math import pi
from IPython.display import display

INPUT_CSV = "cleaned_elss.csv"  # replace with your CSV file
date_col = "Date"
nav_col = "NAV"
group_col = "Scheme Code"
scheme_name_col = "Scheme Name"
isin_col = "ISIN Div Payout/ISIN Growth"


## 4.1.2 Read & basic cleaning
Load CSV, coerce date and NAV to appropriate types, and sort by scheme and date.

In [3]:
df = pd.read_csv(INPUT_CSV, dtype=str)
df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
df[nav_col] = pd.to_numeric(df[nav_col].str.replace(',',''), errors="coerce")
df = df.sort_values([group_col, date_col]).reset_index(drop=True)
print("Loaded data shape:", df.shape)

Loaded data shape: (319355, 6)


## 4.1.3 Datetime features
Extract year, month, day, weekday, quarter, day-of-year, and ISO week-of-year.

In [4]:
df["year"] = df[date_col].dt.year
df["month"] = df[date_col].dt.month
df["day"] = df[date_col].dt.day
df["weekday"] = df[date_col].dt.weekday
df["quarter"] = df[date_col].dt.quarter
df["dayofyear"] = df[date_col].dt.dayofyear
df["weekofyear"] = df[date_col].dt.isocalendar().week.astype("Int64")
display(df.head(2)[[date_col, "year", "month", "weekday", "weekofyear"]])

,Date,year,month,weekday,weekofyear
0,2020-05-04,2020,5,0,19
1,2020-05-05,2020,5,1,19


## 4.1.4 Time index & trend features
Create per-scheme time index and normalized / polynomial trend features.

In [5]:
df["time_index"] = df.groupby(group_col).cumcount() + 1
df["time_index_norm"] = df.groupby(group_col)["time_index"].transform(lambda x: (x - x.mean()) / (x.std() + 1e-9))
df["time_index_sq"] = df["time_index_norm"] ** 2
display(df[[group_col, date_col, "time_index", "time_index_norm", "time_index_sq"]].head(3))

,Scheme Code,Date,time_index,time_index_norm,time_index_sq
0,100067,2020-05-04,1,-1.728504,2.987727
1,100067,2020-05-05,2,-1.723775,2.971401
2,100067,2020-05-06,3,-1.719046,2.955119


## 4.1.5 Previous value, diff & short lags
Compute previous NAV, first difference and short lag features (1,3,7,14,30).

In [6]:
df["nav_prev_1"] = df.groupby(group_col)[nav_col].shift(1)
df["nav_diff_1"] = df[nav_col] - df["nav_prev_1"]
df["nav_diff_1_flag"] = df["nav_prev_1"].isnull().astype(int)
for lag in [1, 3, 7, 14, 30]:
    df[f"nav_lag_{lag}"] = df.groupby(group_col)[nav_col].shift(lag)
display(df.head(3)[[date_col, nav_col, "nav_prev_1", "nav_diff_1", "nav_lag_1"]])

,Date,NAV,nav_prev_1,nav_diff_1,nav_lag_1
0,2020-05-04,70.30,NaN,NaN,NaN
1,2020-05-05,69.73,70.30,-0.57,70.30
2,2020-05-06,70.27,69.73,0.54,69.73


## 4.1.6 Rolling and expanding statistics
Compute rolling mean/std for windows and expanding mean/std for historical context.

In [7]:
for w in [3, 7, 30]:
    df[f"nav_roll_mean_{w}"] = df.groupby(group_col)[nav_col].transform(lambda x: x.rolling(w, min_periods=1).mean())
    df[f"nav_roll_std_{w}"] = df.groupby(group_col)[nav_col].transform(lambda x: x.rolling(w, min_periods=1).std())
df["nav_expanding_mean"] = df.groupby(group_col)[nav_col].transform(lambda x: x.expanding(1).mean())
df["nav_expanding_std"] = df.groupby(group_col)[nav_col].transform(lambda x: x.expanding(1).std())
display(df.head(3)[[nav_col, "nav_roll_mean_7", "nav_roll_std_7", "nav_expanding_mean"]])

,NAV,nav_roll_mean_7,nav_roll_std_7,nav_expanding_mean
0,70.30,70.300,NaN,70.300
1,69.73,70.015,0.403051,70.015
2,70.27,70.100,0.320780,70.100


## 4.1.7 Frequency inference & seasonality features
Infer sampling frequency per scheme and generate month cyclical and annual Fourier features; add seasonal lags.

In [8]:
def infer_freq(g):
    d = g[date_col].dropna().sort_values()
    if len(d) < 2:
        return np.nan
    diffs = d.diff().dt.days.dropna()
    if len(diffs) == 0:
        return np.nan
    return diffs.median()

median_gaps = df.groupby(group_col).apply(infer_freq).rename("median_gap_days")
df = df.join(median_gaps, on=group_col)
df["_is_monthly_like"] = (df["median_gap_days"] >= 25).astype(int)
df["t_days"] = df.groupby(group_col)[date_col].transform(lambda x: (x - x.min()).dt.days)

df["month_sin"] = np.sin(2 * pi * ((df["month"] - 1) / 12))
df["month_cos"] = np.cos(2 * pi * ((df["month"] - 1) / 12))
period_ann = 365.25
for k in [1, 2]:
    df[f"fourier_ann_sin_{k}"] = np.sin(2 * pi * k * df["t_days"] / period_ann)
    df[f"fourier_ann_cos_{k}"] = np.cos(2 * pi * k * df["t_days"] / period_ann)
for lag in [7, 30, 365]:
    df[f"nav_lag_{lag}"] = df.groupby(group_col)[nav_col].shift(lag)
for lag in [12, 24]:
    df[f"nav_lag_month_{lag}"] = df.groupby(group_col)[nav_col].shift(lag)
display(df.head(2)[[date_col, "median_gap_days", "_is_monthly_like", "month_sin", "fourier_ann_sin_1"]])

C:\Users\ASUS\AppData\Local\Temp\ipykernel_13988\1331063637.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  median_gaps = df.groupby(group_col).apply(infer_freq).rename("median_gap_days")


,Date,median_gap_days,_is_monthly_like,month_sin,fourier_ann_sin_1
0,2020-05-04,1.0,0,0.866025,0.000000
1,2020-05-05,1.0,0,0.866025,0.017202


## 4.1.8 Group-level aggregates
Compute per-scheme mean, std, min and max NAV and join back to the main DataFrame.

In [9]:
grp = df.groupby(group_col)[nav_col].agg(["mean", "std", "min", "max"]).rename(columns={
    "mean": "scheme_nav_mean",
    "std": "scheme_nav_std",
    "min": "scheme_nav_min",
    "max": "scheme_nav_max"
})
df = df.join(grp, on=group_col)
display(df.head(2)[[group_col, "scheme_nav_mean", "scheme_nav_std"]])

,Scheme Code,scheme_nav_mean,scheme_nav_std
0,100067,91.323033,8.354307
1,100067,91.323033,8.354307


## 4.1.9 Price-derived features (optional)
If sale and repurchase price columns exist, compute spread, spread_pct_nav and price range features.

In [10]:
sale_col = "Sale Price"
rep_col = "Repurchase Price"
if sale_col in df.columns and rep_col in df.columns:
    df[sale_col] = pd.to_numeric(df[sale_col].str.replace(',',''), errors="coerce")
    df[rep_col] = pd.to_numeric(df[rep_col].str.replace(',',''), errors="coerce")
    df["spread"] = df[sale_col] - df[rep_col]
    df["spread_pct_nav"] = df["spread"] / df[nav_col].replace({0: np.nan})
if {nav_col, sale_col, rep_col}.issubset(df.columns):
    df["price_max"] = df[[nav_col, sale_col, rep_col]].max(axis=1)
    df["price_min"] = df[[nav_col, sale_col, rep_col]].min(axis=1)
    df["price_range"] = df["price_max"] - df["price_min"]
cols_to_show = [c for c in [sale_col, rep_col, "spread", "price_range"] if c in df.columns]
if cols_to_show:
    display(df.head(2)[cols_to_show])

## 4.1.10 Categorical encodings
Convert scheme code, scheme name and ISIN to categorical codes for modeling.

In [11]:
df["scheme_code_cat"] = df[group_col].astype("category").cat.codes
df["scheme_name_cat"] = df[scheme_name_col].astype("category").cat.codes if scheme_name_col in df.columns else -1
if isin_col in df.columns:
    df["isin_code_cat"] = df[isin_col].astype("category").cat.codes
display(df.head(2)[[group_col, "scheme_code_cat", "scheme_name_cat"]])

,Scheme Code,scheme_code_cat,scheme_name_cat
0,100067,0,13
1,100067,0,13


## 4.1.11 Missing flags
Create binary flags for missing NAV and optional price columns.

In [12]:
df["nav_missing_flag"] = df[nav_col].isnull().astype(int)
if sale_col in df.columns:
    df["sale_missing_flag"] = df[sale_col].isnull().astype(int)
if rep_col in df.columns:
    df["repurchase_missing_flag"] = df[rep_col].isnull().astype(int)
display(df.head(2)[[nav_col, "nav_missing_flag"]])

,NAV,nav_missing_flag
0,70.30,0
1,69.73,0


## 4.1.12 Targets
Create next-step targets for supervised learning and drop rows without a valid next target.

In [13]:
df["nav_next"] = df.groupby(group_col)[nav_col].shift(-1)
df["target_next_return"] = (df["nav_next"] - df[nav_col]) / df[nav_col].replace({0: np.nan})
df["target_up"] = (df["target_next_return"] > 0).astype(int)
df = df[~df["nav_next"].isnull()].copy()
print("Shape after creating targets:", df.shape)

Shape after creating targets: (319086, 55)


## 4.1.13 Sanity checks & preview
Quick shape, missing-value summary and sample preview for first scheme.

In [14]:
print(df.shape)
print('\nTop columns:', df.columns.tolist()[:80])
print('\nMissing value counts (top 30):')
print(df.isnull().sum().sort_values(ascending=False).head(30))

first_scheme = df[group_col].dropna().unique()[0]
display(df[df[group_col] == first_scheme].head(12)[[
    date_col, nav_col, "nav_prev_1", "nav_diff_1", "nav_lag_1", "nav_roll_mean_7",
    "month", "month_sin", "month_cos", "fourier_ann_sin_1", "fourier_ann_cos_1",
    "nav_expanding_mean", "nav_next", "target_next_return"
]])

(319086, 55)

Top columns: ['Scheme Code', 'Scheme Name', 'ISIN Div Payout/ISIN Growth', 'ISIN Div Reinvestment', 'NAV', 'Date', 'year', 'month', 'day', 'weekday', 'quarter', 'dayofyear', 'weekofyear', 'time_index', 'time_index_norm', 'time_index_sq', 'nav_prev_1', 'nav_diff_1', 'nav_diff_1_flag', 'nav_lag_1', 'nav_lag_3', 'nav_lag_7', 'nav_lag_14', 'nav_lag_30', 'nav_roll_mean_3', 'nav_roll_std_3', 'nav_roll_mean_7', 'nav_roll_std_7', 'nav_roll_mean_30', 'nav_roll_std_30', 'nav_expanding_mean', 'nav_expanding_std', 'median_gap_days', '_is_monthly_like', 't_days', 'month_sin', 'month_cos', 'fourier_ann_sin_1', 'fourier_ann_cos_1', 'fourier_ann_sin_2', 'fourier_ann_cos_2', 'nav_lag_365', 'nav_lag_month_12', 'nav_lag_month_24', 'scheme_nav_mean', 'scheme_nav_std', 'scheme_nav_min', 'scheme_nav_max', 'scheme_code_cat', 'scheme_name_cat', 'isin_code_cat', 'nav_missing_flag', 'nav_next', 'target_next_return', 'target_up']

Missing value counts (top 30):
ISIN Div Reinvestment          195635

,Date,NAV,nav_prev_1,nav_diff_1,nav_lag_1,nav_roll_mean_7,month,month_sin,month_cos,fourier_ann_sin_1,fourier_ann_cos_1,nav_expanding_mean,nav_next,target_next_return
0,2020-05-04,70.30,NaN,NaN,NaN,70.300000,5,0.866025,-0.5,0.000000,1.000000,70.300000,69.73,-0.008108
1,2020-05-05,69.73,70.30,-0.57,70.30,70.015000,5,0.866025,-0.5,0.017202,0.999852,70.015000,70.27,0.007744
2,2020-05-06,70.27,69.73,0.54,69.73,70.100000,5,0.866025,-0.5,0.034398,0.999408,70.100000,69.70,-0.008112
3,2020-05-07,69.70,70.27,-0.57,70.27,70.000000,5,0.866025,-0.5,0.051584,0.998669,70.000000,69.73,0.000430
4,2020-05-08,69.73,69.70,0.03,69.70,69.946000,5,0.866025,-0.5,0.068755,0.997634,69.946000,69.44,-0.004159
5,2020-05-11,69.44,69.73,-0.29,69.73,69.861667,5,0.866025,-0.5,0.120126,0.992759,69.861667,68.23,-0.017425
6,2020-05-12,68.23,69.44,-1.21,69.44,69.628571,5,0.866025,-0.5,0.137185,0.990545,69.628571,69.60,0.020079
7,2020-05-13,69.60,68.23,1.37,68.23,69.528571,5,0.866025,-0.5,0.154204,0.988039,69.625000,68.59,-0.014511
8,2020-05-14,68.59,69.60,-1.01,69.60,69.365714,5,0.866025,-0.5,0.171177,0.985240,69.510000,68.35,-0.003499
9,2020-05-15,68.35,68.59,-0.24,68.59,69.091429,5,0.866025,-0.5,0.188099,0.982150,69.394000,66.13,-0.032480


## 4.1.14 Save engineered dataset
Save the engineered features to CSV for use in the modelling section.

In [15]:
OUT_CSV = "engineered_features_elss.csv"
df.to_csv(OUT_CSV, index=False)
print("Saved engineered features to:", OUT_CSV)

Saved engineered features to: engineered_features_elss.csv


# **4.2 Feature Selection**

## **Overview**
Feature selection is performed after feature extraction and before final model training.  
Its purpose is to remove redundant, highly correlated, or uninformative features to improve model performance, reduce overfitting, and simplify interpretation.  
Since the engineered dataset includes many time-series features (lags, rolling windows, trend terms, Fourier components, etc.), a structured selection strategy is essential.

---

## **4.2.1 Rationale**
Without feature selection:

- Models may overfit to noise  
- Training time increases  
- Interpretability decreases  
- Collinearity can destabilize linear models  

Therefore, selecting the most relevant and stable predictors is crucial.

---

## **4.2.2 Procedure**

### **1. Remove near-constant features (low variance)**
Columns with almost no variability across rows are removed, as they provide no useful information.

### **2. Filter out features with high missingness**
Features with a very high proportion of missing values (e.g., > 80%) are discarded because they lack usable signal and can distort imputation.

### **3. Correlation filtering**
A correlation matrix is computed for numeric features.  
For pairs with extremely high correlation (e.g., |corr| > 0.95), only one feature is retained.  
This reduces multicollinearity and stabilizes model training.

### **4. Univariate statistical scoring**
Depending on the task:

- Regression → Mutual Information, ANOVA F-tests  
- Classification → Chi-square, Mutual Information  

Low-scoring features are removed as they contribute little predictive power.

### **5. Model-based feature importance**
Tree-based models (Random Forest, XGBoost) are trained on the training set.  
Feature importance scores are extracted, and low-importance features are removed.  
These models capture nonlinear relationships and interactions, making their rankings highly informative.


## **4.2.3 Practical Notes**

- **Perform selection only on training folds (time-aware CV).**  
  This avoids information leakage from test periods.

- **Keep imputation flags**, as missingness can itself be predictive.

- **Document thresholds** used (missingness %, correlation cutoff, importance thresholds).

- **Include visual summaries** such as:  
  - Feature importance plots  
  - Correlation heatmaps  

---





